In [1]:
import os
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# I. Subway Hourly Ridership Dataset

In [61]:
URLS = {
    "25": "5wq4-mkjj",
    "24": "wujg-7c2s",
}

In [63]:
def collect_daily_ridership_by_station_id(
    year: str,
    url: str
    ):
    BASE_URL = f"https://data.ny.gov/resource/{url}.json"

    # Optional but recommended if you make many requests:
    APP_TOKEN = None  # e.g. "YOUR_SOCRATA_APP_TOKEN"

    # Pagination
    PAGE_SIZE = 50000
    offset = 0

    # Output
    OUT_CSV = f"data/mta_daily_station_ridership_by_station_id_{year}.csv"

    # Your working query shape (using Socrata backticks as in the data viewer)
    BASE_QUERY = (
        "SELECT "
        "`station_complex_id`, "
        "sum(`ridership`) AS daily_ridership, "
        "date_trunc_ymd(`transit_timestamp`) AS by_day_transit_timestamp "
        f"WHERE `transit_timestamp` >= '20{year}-01-01' "
        f"AND `transit_timestamp` < '20{int(year)+1}-01-01' "
        "GROUP BY `station_complex_id`, date_trunc_ymd(`transit_timestamp`) "
        "ORDER BY by_day_transit_timestamp, `station_complex_id`"
    )

    def make_session(app_token=None):
        s = requests.Session()

        # Retry on transient errors + timeouts
        retry = Retry(
            total=8,
            connect=8,
            read=8,
            backoff_factor=1.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=("GET",),
            raise_on_status=False,
            respect_retry_after_header=True,
        )
        adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
        s.mount("https://", adapter)
        s.mount("http://", adapter)

        if app_token:
            s.headers.update({"X-App-Token": app_token})

        return s

    session = make_session(APP_TOKEN)
    
    print(f"Starting data collection for year 20{year}...")

    # If file exists, resume by counting rows already written (optional resume)
    if os.path.exists(OUT_CSV):
        existing_rows = sum(1 for _ in open(OUT_CSV, "r", encoding="utf-8")) - 1  # minus header
        if existing_rows > 0:
            offset = (existing_rows // PAGE_SIZE) * PAGE_SIZE
            print(f"Found existing {existing_rows:,} rows in {OUT_CSV}. Resuming at offset={offset}.")

    first_write = not os.path.exists(OUT_CSV)

    while True:
        
        paged_query = f"{BASE_QUERY} LIMIT {PAGE_SIZE} OFFSET {offset}"

        try:
            r = session.get(
                BASE_URL,
                params={"$query": paged_query},
                timeout=(30, 240),  # (connect timeout, read timeout)
            )
        except requests.exceptions.ReadTimeout:
            # Extra backoff on top of urllib3 retries, just in case
            print("ReadTimeout hit; sleeping 10s and retrying same page...")
            time.sleep(10)
            continue

        # If still not OK, show body and stop (you can restart; it will resume)
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}\nURL: {r.url}\nResponse:\n{r.text}")

        batch = r.json()
        if not batch:
            print("No more rows; done.")
            break

        df = pd.DataFrame(batch)

        # Normalize types/column names
        df.rename(columns={"by_day_transit_timestamp": "service_date"}, inplace=True)
        df["service_date"] = pd.to_datetime(df["service_date"]).dt.date
        df["daily_ridership"] = pd.to_numeric(df["daily_ridership"], errors="coerce").fillna(0).astype(int)

        # Append to CSV incrementally
        df.to_csv(OUT_CSV, mode="a", index=False, header=first_write)
        first_write = False

        offset += PAGE_SIZE
        print(f"Wrote {len(df):,} rows (next offset={offset})")

    print(f"Saved: {OUT_CSV}")


In [64]:
for y, u in URLS.items():
    collect_daily_ridership_by_station_id(year=y, url=u)

Starting data collection for year 2025...
Wrote 50,000 rows (next offset=50000)
Wrote 50,000 rows (next offset=100000)
Wrote 50,000 rows (next offset=150000)
Wrote 4,828 rows (next offset=200000)
No more rows; done.
Saved: data/mta_daily_station_ridership_by_station_id_25.csv
Starting data collection for year 2024...
Wrote 50,000 rows (next offset=50000)
Wrote 50,000 rows (next offset=100000)
Wrote 50,000 rows (next offset=150000)
Wrote 6,366 rows (next offset=200000)
No more rows; done.
Saved: data/mta_daily_station_ridership_by_station_id_24.csv


In [71]:
df_25 = pd.read_csv("data/raw/mta_daily_station_ridership_by_station_id_25.csv")

In [72]:
df_25.columns

Index(['station_complex_id', 'daily_ridership', 'service_date'], dtype='object')

In [73]:
# check for duplicates
df_25.duplicated(subset=["station_complex_id", "service_date"]).sum()

0

In [74]:
df_25.shape

(154828, 3)

In [75]:
df_24.shape

(154902, 3)

In [76]:
# restict 25 df to dates in 25
df_25 = df_25[
    (df_25["service_date"] >= "2025-01-01") &
    (df_25["service_date"] < "2026-01-01")
]

df_25.shape

(154828, 3)

In [77]:
df_24 = pd.read_csv("data/raw/mta_daily_station_ridership_by_station_id_24.csv")

In [78]:
# restict 25 df to dates in 25
df_24 = df_24[
    (df_24["service_date"] >= "2024-01-01") &
    (df_24["service_date"] < "2025-01-01")
]

df_24.shape

(156366, 3)

In [79]:
# drop dups
df_24 = df_24.drop_duplicates(subset=["station_complex_id", "service_date"])

df_24.shape

(156366, 3)

In [80]:
df_24['service_date'] = pd.to_datetime(df_24['service_date']).dt.date
df_25['service_date'] = pd.to_datetime(df_25['service_date']).dt.date

In [81]:
# find out if any dates are missing
all_dates_24 = pd.date_range(start="2024-01-01", end="2024-12-31").date
all_dates_25 = pd.date_range(start="2025-01-01", end="2025-12-31").date

# find dates not in 2024
missing_dates_24 = set(all_dates_24) - set(df_24["service_date"].unique())
missing_dates_24

set()

In [82]:
missing_dates_25 = set(all_dates_25) - set(df_25["service_date"].unique())
missing_dates_25

set()

In [83]:
all_stations_24 = set(df_24["station_complex_id"].unique())
all_stations_25 = set(df_25["station_complex_id"].unique())

print(all_stations_24 - all_stations_25)
print(all_stations_25 - all_stations_24)

set()
set()


In [84]:
# write back to csv
df_24.to_csv("data/raw/mta_daily_station_ridership_by_station_id_24.csv", index=False)
df_25.to_csv("data/raw/mta_daily_station_ridership_by_station_id_25.csv", index=False)

In [85]:
avg_daily_ridership_24 = df_24.groupby("station_complex_id")["daily_ridership"].mean().reset_index()
avg_daily_ridership_24.rename(columns={"daily_ridership": "avg_daily_ridership_24"}, inplace=True)
avg_daily_ridership_24

,station_complex_id,avg_daily_ridership_24
0,1,9275.732240
1,10,17960.680328
2,100,1845.934426
3,101,7809.797814
4,103,2437.461326
...,...,...
423,97,7625.095628
424,98,4578.292350
425,99,2496.251366
426,TRAM1,4678.377049


In [86]:
avg_daily_ridership_25 = df_25.groupby("station_complex_id")["daily_ridership"].mean().reset_index()
avg_daily_ridership_25.rename(columns={"daily_ridership": "avg_daily_ridership_25"}, inplace=True)
avg_daily_ridership_25

,station_complex_id,avg_daily_ridership_25
0,1,9882.704110
1,10,20218.238356
2,100,1867.865753
3,101,8240.794521
4,103,2779.063014
...,...,...
423,97,8024.134247
424,98,5528.854795
425,99,2874.915068
426,TRAM1,4153.829670


In [87]:
# merge avg daily riderships
avg_daily_ridership = pd.merge(
    avg_daily_ridership_24,
    avg_daily_ridership_25,
    on="station_complex_id",
    how="inner"
)
avg_daily_ridership

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25
0,1,9275.732240,9882.704110
1,10,17960.680328,20218.238356
2,100,1845.934426,1867.865753
3,101,7809.797814,8240.794521
4,103,2437.461326,2779.063014
...,...,...,...
423,97,7625.095628,8024.134247
424,98,4578.292350,5528.854795
425,99,2496.251366,2874.915068
426,TRAM1,4678.377049,4153.829670


In [102]:
avg_daily_ridership[avg_daily_ridership['station_complex_id'].str.startswith('50')]

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25,change_pct,change
342,50,3266.254795,3672.491713,0.124374,406.236918
343,501,5555.836066,5487.698630,-0.012264,-68.137435
344,502,374.606557,431.651099,0.152279,57.044542


In [88]:
avg_daily_ridership['change_pct'] = (
    (avg_daily_ridership['avg_daily_ridership_25'] - avg_daily_ridership['avg_daily_ridership_24'])
    / avg_daily_ridership['avg_daily_ridership_24']
)
avg_daily_ridership['change'] = avg_daily_ridership['avg_daily_ridership_25'] - avg_daily_ridership['avg_daily_ridership_24']

avg_daily_ridership

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25,change_pct,change
0,1,9275.732240,9882.704110,0.065437,606.971869
1,10,17960.680328,20218.238356,0.125694,2257.558028
2,100,1845.934426,1867.865753,0.011881,21.931327
3,101,7809.797814,8240.794521,0.055187,430.996706
4,103,2437.461326,2779.063014,0.140147,341.601688
...,...,...,...,...,...
423,97,7625.095628,8024.134247,0.052332,399.038618
424,98,4578.292350,5528.854795,0.207624,950.562445
425,99,2496.251366,2874.915068,0.151693,378.663702
426,TRAM1,4678.377049,4153.829670,-0.112122,-524.547379


In [89]:
avg_daily_ridership.sort_values(by='change_pct', ascending=True)

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25,change_pct,change
412,86,792.988539,488.548571,-0.383915,-304.439967
324,455,3926.131148,3158.841096,-0.195432,-767.290052
410,84,1988.751397,1603.128134,-0.193902,-385.623263
326,457,4076.892857,3435.449315,-0.157336,-641.443542
320,450,17184.680328,15257.487671,-0.112146,-1927.192657
...,...,...,...,...,...
318,449,5771.893443,9442.821918,0.636001,3670.928475
80,198,2496.387978,4314.090411,0.728133,1817.702433
411,85,1260.511111,2387.927577,0.894412,1127.416465
409,83,1557.193989,3062.586301,0.966734,1505.392312


## I.I Merging with station data

In [112]:
complexes_url = "https://data.ny.gov/api/views/5f5g-n3cz/rows.csv?accessType=DOWNLOAD"

complexes_df = pd.read_csv(complexes_url)

complexes_df.head()

,Complex ID,Is Complex,Number Of Stations In Complex,Stop Name,Display Name,Constituent Station Names,Station IDs,GTFS Stop IDs,Borough,CBD,Daytime Routes,Structure Type,Latitude,Longitude,ADA,ADA Notes
0,398,False,1,77 St,77 St (6),77 St,398,627,M,False,6,Subway,40.773620,-73.959874,0,NaN
1,399,False,1,68 St-Hunter College,68 St-Hunter College (6),68 St-Hunter College,399,628,M,False,6,Subway,40.768141,-73.963870,1,NaN
2,403,False,1,33 St,33 St (6),33 St,403,632,M,True,6,Subway,40.746081,-73.982076,0,NaN
3,404,False,1,28 St,28 St (6),28 St,404,633,M,True,6,Subway,40.743070,-73.984264,2,Downtown only
4,405,False,1,23 St-Baruch College,23 St-Baruch College (6),23 St-Baruch College,405,634,M,True,6,Subway,40.739864,-73.986599,1,NaN


In [113]:
# turn the col names to snake case
complexes_df.columns = [col.lower().replace(" ", "_") for col in complexes_df.columns]

In [114]:
complexes_ids = set(complexes_df["complex_id"].unique())
ids = set(avg_daily_ridership["station_complex_id"].unique())

# check if all ids in avg_daily_ridership are in complexes_df
ids - complexes_ids

{'1',
 '10',
 '100',
 '101',
 '103',
 '107',
 '108',
 '109',
 '110',
 '111',
 '113',
 '114',
 '118',
 '119',
 '120',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '129',
 '13',
 '130',
 '131',
 '133',
 '134',
 '135',
 '136',
 '137',
 '138',
 '14',
 '141',
 '143',
 '144',
 '145',
 '146',
 '147',
 '149',
 '150',
 '151',
 '152',
 '153',
 '154',
 '155',
 '156',
 '157',
 '158',
 '159',
 '16',
 '160',
 '162',
 '164',
 '165',
 '167',
 '168',
 '169',
 '17',
 '173',
 '175',
 '176',
 '177',
 '179',
 '180',
 '181',
 '182',
 '183',
 '185',
 '186',
 '187',
 '188',
 '189',
 '190',
 '191',
 '192',
 '193',
 '194',
 '195',
 '196',
 '197',
 '198',
 '199',
 '2',
 '20',
 '200',
 '201',
 '202',
 '203',
 '204',
 '205',
 '206',
 '207',
 '208',
 '209',
 '210',
 '211',
 '212',
 '213',
 '214',
 '215',
 '216',
 '217',
 '218',
 '22',
 '220',
 '221',
 '222',
 '223',
 '224',
 '225',
 '228',
 '231',
 '232',
 '234',
 '235',
 '236',
 '237',
 '238',
 '240',
 '241',
 '242',
 '243',
 '244',
 '245',
 '246',
 '247',
 '2

In [115]:
# dedup complexes_df by complex_id
complexes_df = complexes_df.drop_duplicates(subset=["display_name"])

complexes_df

,complex_id,is_complex,number_of_stations_in_complex,stop_name,display_name,constituent_station_names,station_ids,gtfs_stop_ids,borough,cbd,daytime_routes,structure_type,latitude,longitude,ada,ada_notes
0,398,False,1,77 St,77 St (6),77 St,398,627,M,False,6,Subway,40.773620,-73.959874,0,NaN
1,399,False,1,68 St-Hunter College,68 St-Hunter College (6),68 St-Hunter College,399,628,M,False,6,Subway,40.768141,-73.963870,1,NaN
2,403,False,1,33 St,33 St (6),33 St,403,632,M,True,6,Subway,40.746081,-73.982076,0,NaN
3,404,False,1,28 St,28 St (6),28 St,404,633,M,True,6,Subway,40.743070,-73.984264,2,Downtown only
4,405,False,1,23 St-Baruch College,23 St-Baruch College (6),23 St-Baruch College,405,634,M,True,6,Subway,40.739864,-73.986599,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
440,134,False,1,Sutter Av,Sutter Av (L),Sutter Av,134,L25,Bk,False,L,Elevated,40.669367,-73.901975,0,NaN
441,135,False,1,Livonia Av,Livonia Av (L),Livonia Av,135,L26,Bk,False,L,Elevated,40.664038,-73.900571,1,NaN
442,136,False,1,New Lots Av,New Lots Av (L),New Lots Av,136,L27,Bk,False,L,Elevated,40.658733,-73.899232,0,NaN
443,137,False,1,East 105 St,East 105 St (L),East 105 St,137,L28,Bk,False,L,At Grade,40.650573,-73.899485,0,NaN


In [116]:
cols_for_merge = [
    "complex_id",
    "display_name",
    "latitude",
    "longitude",
    "borough"
]

complexes_df = complexes_df[cols_for_merge]

complexes_df

,complex_id,display_name,latitude,longitude,borough
0,398,77 St (6),40.773620,-73.959874,M
1,399,68 St-Hunter College (6),40.768141,-73.963870,M
2,403,33 St (6),40.746081,-73.982076,M
3,404,28 St (6),40.743070,-73.984264,M
4,405,23 St-Baruch College (6),40.739864,-73.986599,M
...,...,...,...,...,...
440,134,Sutter Av (L),40.669367,-73.901975,Bk
441,135,Livonia Av (L),40.664038,-73.900571,Bk
442,136,New Lots Av (L),40.658733,-73.899232,Bk
443,137,East 105 St (L),40.650573,-73.899485,Bk


In [117]:
complexes_df[complexes_df["borough"] == "SI"]

,complex_id,display_name,latitude,longitude,borough
59,501,St George (SIR),40.643748,-74.073643,SI
60,502,Tompkinsville (SIR),40.636949,-74.074835,SI
61,503,Stapleton (SIR),40.627915,-74.075162,SI
62,504,Clifton (SIR),40.621319,-74.071402,SI
63,505,Grasmere (SIR),40.603117,-74.084087,SI
64,506,Old Town (SIR),40.596612,-74.087368,SI
65,507,Dongan Hills (SIR),40.588849,-74.096090,SI
66,508,Jefferson Av (SIR),40.583591,-74.103338,SI
67,509,Grant City (SIR),40.578965,-74.109704,SI
68,510,New Dorp (SIR),40.573480,-74.117210,SI


In [120]:
complexes_df['complex_id'] = complexes_df['complex_id'].astype(str)

/var/folders/jd/5wy1jytx2pg8j4jr12tl0k_m0000gq/T/ipykernel_13461/3912323095.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  complexes_df['complex_id'] = complexes_df['complex_id'].astype(str)


In [121]:
# merge
final_df = pd.merge(
    avg_daily_ridership,
    complexes_df,
    left_on="station_complex_id",
    right_on="complex_id",
    how="left"
)

final_df

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25,change_pct,change,complex_id,display_name,latitude,longitude,borough
0,1,9275.732240,9882.704110,0.065437,606.971869,1,"Astoria-Ditmars Blvd (N,W)",40.775036,-73.912034,Q
1,10,17960.680328,20218.238356,0.125694,2257.558028,10,"49 St (N,R,W)",40.759901,-73.984139,M
2,100,1845.934426,1867.865753,0.011881,21.931327,100,"Hewes St (M,J)",40.706870,-73.953431,Bk
3,101,7809.797814,8240.794521,0.055187,430.996706,101,"Marcy Av (M,J,Z)",40.708359,-73.957757,Bk
4,103,2437.461326,2779.063014,0.140147,341.601688,103,"Bowery (J,Z)",40.720280,-73.993915,M
...,...,...,...,...,...,...,...,...,...,...
423,97,7625.095628,8024.134247,0.052332,399.038618,97,"Myrtle Av (M,J,Z)",40.697207,-73.935657,Bk
424,98,4578.292350,5528.854795,0.207624,950.562445,98,"Flushing Av (M,J)",40.700260,-73.941126,Bk
425,99,2496.251366,2874.915068,0.151693,378.663702,99,"Lorimer St (M,J)",40.703869,-73.947408,Bk
426,TRAM1,4678.377049,4153.829670,-0.112122,-524.547379,NaN,NaN,NaN,NaN,NaN


In [122]:
final_df.sort_values(by='change_pct', ascending=False)

,station_complex_id,avg_daily_ridership_24,avg_daily_ridership_25,change_pct,change,complex_id,display_name,latitude,longitude,borough
30,138,1514.270492,6206.767123,3.098850,4692.496631,138,Canarsie-Rockaway Pkwy (L),40.646654,-73.901850,Bk
409,83,1557.193989,3062.586301,0.966734,1505.392312,83,"Woodhaven Blvd (J,Z)",40.693879,-73.851576,Q
411,85,1260.511111,2387.927577,0.894412,1127.416465,85,"75 St-Elderts Ln (J,Z)",40.691324,-73.867139,Q
80,198,2496.387978,4314.090411,0.728133,1817.702433,198,Howard Beach-JFK Airport (A),40.660476,-73.830301,Q
318,449,5771.893443,9442.821918,0.636001,3670.928475,449,111 St (7),40.751730,-73.855334,Q
...,...,...,...,...,...,...,...,...,...,...
320,450,17184.680328,15257.487671,-0.112146,-1927.192657,450,103 St-Corona Plaza (7),40.749865,-73.862700,Q
326,457,4076.892857,3435.449315,-0.157336,-641.443542,457,52 St (7),40.744149,-73.912549,Q
410,84,1988.751397,1603.128134,-0.193902,-385.623263,84,85 St-Forest Pkwy (J),40.692435,-73.860010,Q
324,455,3926.131148,3158.841096,-0.195432,-767.290052,455,69 St (7),40.746325,-73.896403,Q


In [123]:
final_df['borough'] = final_df['borough'].map({
    'M': 'Manhattan',
    'Bk': 'Brooklyn',
    'Q': 'Queens',
    'Bx': 'Bronx',
})


In [124]:
final_df.to_csv("data/processed/mta_subway_ridership_change.csv", index=False)